# WX-1 자동 학습 (Kaggle One-Click)

**Save Version → Run All** 한 번이면 끝. 탭 닫아도 백그라운드로 계속 돈다.

## 실행 전 필수 설정 (1분이면 됨)

1. 우측 **Settings** → **Accelerator: GPU T4 x2**
   - P100 은 sm_60 이라现代 PyTorch(2.7+) 에서 커널이 죽는다. **반드시 T4.**
2. 우측 **Settings** → **Internet: On**
   - 꺼져 있으면 `pip install` 과 `git clone` 이 둘 다 죽는다.
3. **Save Version** 실행 → 완료 후 **Output** 탭에서 산출물 수신

## 왜 Kaggle 인지

| | Colab 무료 | Kaggle 무료 |
|---|---|---|
| 시스템 RAM | 12.7 GB | **29 GB** |
| GPU | T4 16GB | T4 x2 (32GB) |
| 7B fp16 병합 (CPU, 15.2GB) | OOM | 통과 |
| 탭 닫고 계속 실행 | 불가 | 가능 |

> ⚠️ **디스크 주의** — 병합 15.2GB + f16 GGUF 15.2GB = 피크 약 30GB.
> 셀 11 의 가드가 미리 검사한다. 부족하면 거기서 멈추고 안내한다.


In [ ]:
# === 설정 ===
GITHUB_USER = "ijeonghu522-stack"
GITHUB_REPO = "wuxicode"
TRAIN_MODEL = "wx-1-coder"

# Kaggle RAM 29GB 라 7B 병합(15.2GB) 이 들어간다.
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

MAX_SAMPLES = 5000
EPOCHS = 3
SEQ_LEN = 4096

# 최종 산출물만 working 에 남긴다 (오토세이브 대상).
WORK = "/kaggle/working"
# 중간물(병합 모델 15.2GB, f16 GGUF 15.2GB) 은 임시 디스크에 격리한다.
TMP = "/kaggle/temp"

print("WX-1 설정")
print(f"  repo    : {GITHUB_USER}/{GITHUB_REPO}")
print(f"  model   : {TRAIN_MODEL}")
print(f"  base    : {BASE_MODEL}")
print(f"  samples : {MAX_SAMPLES}  epochs={EPOCHS}  seq_len={SEQ_LEN}")
print(f"  work    : {WORK}")
print(f"  temp    : {TMP}")


In [ ]:
# === 환경 확인 ===
# T4 가 아니라면 여기서 바로 잡는다. P100(sm_60) 은 커널이 죽으므로 중단.
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import re
import shutil
import subprocess

info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

if not info:
    raise SystemExit("[FAIL] GPU 없음. Settings → Accelerator → GPU T4 x2 로 바꾸세요.")

caps = re.findall(r"([0-9.]+)", info)
names = [line.split(",")[0].strip() for line in info.splitlines()]
print("GPU:", ", ".join(names))
print("cc :", caps)

if caps and all(float(c) < 7.0 for c in caps):
    raise SystemExit(
        "[FAIL] P100(sm_60) 입니다. T4(sm_75) 로 바꾸세요.\n       P100 은 PyTorch 2.7+ 에서 지원이 끊겨 커널이 조용히 죽습니다."
    )

ram_gb = shutil.disk_usage("/").total / (1024 ** 3)
free_gb = shutil.disk_usage("/").free / (1024 ** 3)
print(f"디스크 전체 {ram_gb:.0f} GB / 여유 {free_gb:.0f} GB")

try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_gb = int(line.split()[1]) / 1024 / 1024
                print(f"시스템 RAM {total_gb:.1f} GB")
                if total_gb < 18:
                    raise SystemExit(
                        f"[FAIL] RAM {total_gb:.1f}GB < 18GB. 7B fp16 병합 불가. "
                        "Settings 에서 GPU 를 선택했는지 확인하세요."
                    )
                break
except FileNotFoundError:
    print("(MemInfo 없음 — RAM 검사 건너뜀)")


In [ ]:
# === 빌드 도구 ===
# llama.cpp 를 직접 빌드하므로 cmake / g++ 가 필요하다.
!apt-get update -qq 2>/dev/null || true
!apt-get install -y -qq cmake g++ ninja-build 2>/dev/null || true

!cmake --version | head -1
!g++ --version | head -1


In [ ]:
!pip install -q --upgrade "numpy>=2.0" "transformers>=4.56" "trl>=0.12" \
    "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" \
    "datasets>=3.0" "pyyaml>=6.0"


In [ ]:
# 런타임 재시작: 필요하지 않습니다.
# 학습/병합/변환이 모두 `!python ...` 서브프로세스로 실행되므로
# 새로 설치한 numpy/transformers/trl 을 깨끗한 프로세스에서 로드합니다.
print("=" * 56)
print("  런타임 재시작 불필요 — 이후 셀은 별도 프로세스에서 실행됩니다.")
print("=" * 56)


In [ ]:
# === 작업 디렉터리 ===
import os

for d in (WORK, TMP):
    os.makedirs(d, exist_ok=True)

# repo 는 TMP 에 둔다. WORK 는 오토세이브 대상이라 최종 산출물만
# 남겨야 한다 (repo + datasets + output 을 얹으면 20GB 를 넘긴다).
repo = f"{TMP}/wuxicode"
print("repo :", repo)
print("work :", WORK, " (오토세이브 — 최종 산출물만)")
print("temp :", TMP, " (중간물, 세션 종료 시 소멸)")


In [ ]:
!rm -rf {repo}
!git clone --depth 1 https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {repo}
%cd {repo}
!ls training/


In [ ]:
# train_qlora.py 를 최신 TRL(SFTConfig) 스타일로 패치
!python training/fix_train_qlora.py


In [ ]:
# 데이터셋 생성 (설정 셀의 MAX_SAMPLES 사용)
!python training/auto_build_datasets.py {MAX_SAMPLES}
!wc -l datasets/wx_coder.jsonl


In [ ]:
# config 를 노트북 설정값에 맞춰 조정 (들여쓰기 보존)
import re

path = "training/config/wx-1-coder.yaml"
s = open(path, encoding="utf-8").read()


def set_key(text, key, value):
    # 원래 들여쓰기를 그대로 유지해야 YAML 이 깨지지 않는다
    pat = r"^(\s*)" + re.escape(key) + r":.*$"
    return re.sub(pat, lambda m: f"{m.group(1)}{key}: {value}", text, flags=re.MULTILINE)


s = set_key(s, "base_model", BASE_MODEL)
s = set_key(s, "max_samples", MAX_SAMPLES)
s = set_key(s, "num_train_epochs", EPOCHS)
s = set_key(s, "max_seq_length", SEQ_LEN)

open(path, "w", encoding="utf-8").write(s)

import yaml

cfg = yaml.safe_load(s)
print("base_model  :", cfg["base_model"])
print("max_samples :", cfg["dataset"]["max_samples"])
print("epochs      :", cfg["training"]["num_train_epochs"])
print("max_seq_len :", cfg["training"]["max_seq_length"])


In [ ]:
# === 학습 (QLoRA) ===
# 7B 4bit NF4 는 약 4.5GB + activations. T4 16GB 에서 충분.
# device_map="auto" 가 두 GPU 에 쪼개려도 되지만 QLoRA 는 단일 GPU 가
# 예측 가능하므로 0번만 쓴다.
!CUDA_VISIBLE_DEVICES=0 python training/train_qlora.py \
    --config training/config/wx-1-coder.yaml


In [ ]:
# 학습 직후 즉시 백업 (세션 끊겨도 산출물에 남는다)
import shutil
from pathlib import Path

out = Path("output") / f"{TRAIN_MODEL}-lora"
if not (out / "adapter_config.json").exists():
    raise SystemExit(f"[FAIL] {out}/adapter_config.json 없음. 학습 셀 로그를 확인하세요.")

zip_path = shutil.make_archive(str(out), "zip", out)
shutil.copy(zip_path, WORK)
print(f"[OK] LoRA 백업: {WORK}/{Path(zip_path).name}")


In [ ]:
# === 병합 (CPU, fp16) ===
# 메모리: 7B x 2 byte = 약 15.2GB. Kaggle RAM 29GB 라 통과.
merged = f"{TMP}/{TRAIN_MODEL}-merged"
!python training/merge_adapter.py \
    --base {BASE_MODEL} \
    --adapter output/{TRAIN_MODEL}-lora \
    --output {merged} \
    --dtype float16


In [ ]:
# === 디스크 가드 (GGUF 변환 전) ===
# 병합 모델 15.2GB 가 이미 있고, 변환이 f16 GGUF 15.2GB 를 더 써서
# 피크가 약 30GB 까지 오른다. 여기서 미리 확인해서 2시간 뒤에 죽는
# 대신 지금 멈춘다.
import shutil

NEEDED_GB = 17.0
u = shutil.disk_usage(TMP)
free_gb = u.free / (1024 ** 3)

print(f"중간 디스크({TMP})")
print(f"  전체 {u.total / (1024 ** 3):.0f} GB / 여유 {free_gb:.1f} GB")
print(f"GGUF 변환 필요치: 약 {NEEDED_GB:.0f} GB")

if free_gb < NEEDED_GB:
    print("=" * 56)
    print("  [STOP] 디스크 부족. 학습 결과는 이미 백업되어 있다.")
    print(f"  LoRA: {WORK}/{TRAIN_MODEL}-lora.zip")
    print("=" * 56)
    print("  방법 1: Settings 에서 더 큰 디스크 런타임 선택")
    print("  방법 2: BASE_MODEL 을 1.5B 로 낮추고 재실행")
    print("  방법 3: 아래 GGUF 셀만 단독 실행 (중간물은 이미 병합됨)")
    raise SystemExit(1)


In [ ]:
# === llama.cpp ===
!rm -rf {TMP}/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp {TMP}/llama.cpp
!pip install -q -r {TMP}/llama.cpp/requirements.txt
!pip install -q "protobuf>=5.29.0"


In [ ]:
# === GGUF f16 변환 ===
!python {TMP}/llama.cpp/convert_hf_to_gguf.py {TMP}/{TRAIN_MODEL}-merged \
    --outfile {TMP}/{TRAIN_MODEL}-f16.gguf \
    --outtype f16


In [ ]:
# 변환 끝났으면 병합 원본은 더 이상 필요 없다. 15.2GB 회수.
import shutil

shutil.rmtree(f"{TMP}/{TRAIN_MODEL}-merged", ignore_errors=True)
print(f"병합 원본 삭제. 여유: {shutil.disk_usage('/').free / (1024 ** 3):.1f} GB")


In [ ]:
# === Q4_K_M 양자화 ===
!cmake -B {TMP}/llama.cpp/build -S {TMP}/llama.cpp \
    -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null
!cmake --build {TMP}/llama.cpp/build --config Release -j4 --target llama-quantize


In [ ]:
!{TMP}/llama.cpp/build/bin/llama-quantize \
    {TMP}/{TRAIN_MODEL}-f16.gguf \
    {TMP}/{TRAIN_MODEL}-q4_k_m.gguf \
    Q4_K_M


In [ ]:
# f16 GGUF 는 더 이상 필요 없다. 최종본만 working 에 남긴다.
import shutil
from pathlib import Path

f16 = Path(f"{TMP}/{TRAIN_MODEL}-f16.gguf")
q4 = Path(f"{TMP}/{TRAIN_MODEL}-q4_k_m.gguf")

if not q4.exists():
    raise SystemExit(f"[FAIL] {q4} 없음. 양자화 셀 로그를 확인하세요.")

shutil.copy(q4, WORK)
f16.unlink(missing_ok=True)
shutil.rmtree(f"{TMP}/llama.cpp", ignore_errors=True)

size_gb = q4.stat().st_size / (1024 ** 3)
print(f"[OK] {WORK}/{q4.name}  ({size_gb:.2f} GB)")


In [ ]:
# === 산출물 목록 ===
# Kaggle 은 셀 단위 다운로드가 없다.
# Save Version 실행 후 Output 탭(또는 API datasets_output_download)에서 받는다.
from pathlib import Path

print("=" * 56)
print("  산출물 — Save Version 후 Output 탭에서 수신")
print("=" * 56)

total = 0
for p in sorted(Path(WORK).rglob("*")):
    if p.is_file() and (p.suffix in (".gguf", ".zip") or p.suffix == ".safetensors"):
        mb = p.stat().st_size / (1024 ** 2)
        total += mb
        print(f"  {mb:10.1f} MB  {p.relative_to(WORK)}")

print("=" * 56)
print(f"  합계 {total / 1024:.2f} GB")
print("=" * 56)
